In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Initialisation des qubits

<details>
<summary><b>Versions des packages</b></summary>

Le code de cette page a été développé avec les versions suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Lorsqu'un circuit est exécuté sur une unité de traitement quantique (QPU) IBM&reg;, une réinitialisation implicite est généralement insérée au début du circuit pour s'assurer que les qubits sont initialisés à zéro. Cela est contrôlé par le drapeau `init_qubits`, défini comme une [option d'exécution de primitive](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2).

Cependant, le processus de réinitialisation n'est pas parfait, ce qui entraîne des erreurs de préparation d'état. Pour atténuer l'erreur, le système insère également un délai de répétition (ou `rep_delay`) entre les circuits. Chaque backend a un `rep_delay` par défaut différent, mais il est généralement plus long que T1 pour permettre à l'environnement de réinitialiser les qubits. Le `rep_delay` par défaut peut être interrogé en exécutant `backend.default_rep_delay`.

Toutes les QPU IBM utilisent l'exécution à taux de répétition dynamique, ce qui te permet de modifier le `rep_delay` pour chaque job. Les circuits que tu soumet dans un job primitif sont regroupés pour être exécutés sur la QPU. Ces circuits sont exécutés en itérant sur les circuits pour chaque shot demandé ; l'exécution se fait colonne par colonne sur une matrice de circuits et de shots, comme illustré dans la figure suivante.

![La première colonne représente le shot 0. Les circuits sont exécutés dans l'ordre de 0 à 3. La deuxième colonne représente le shot 1. Les circuits sont exécutés dans l'ordre de 0 à 3. Les colonnes restantes suivent le même schéma.](../docs/images/guides/repetition-rate-execution/circuits_shots_matrix1.avif "Matrice d'exécution par colonnes")

Étant donné que `rep_delay` est inséré entre les circuits, chaque shot de l'exécution rencontre ce délai. Par conséquent, réduire le `rep_delay` diminue le temps total d'exécution sur la QPU, mais au prix d'un taux d'erreur de préparation d'état plus élevé, comme on peut le voir dans l'image suivante :

![Cette image montre que lorsque la valeur de `rep_delay` est abaissée, le taux d'erreur de préparation d'état augmente.](../docs/images/guides/repetition-rate-execution/rep_delay.avif "Délai de répétition par rapport au taux d'erreur")

Définir à la fois `rep_delay=0` et `init_qubits=False` « fusionne » essentiellement les circuits, puisque les qubits commenceront dans l'état final du shot précédent.

Note que si les circuits d'un job primitif sont regroupés pour l'exécution sur la QPU, l'ordre d'exécution des circuits des PUBs n'est pas garanti. Ainsi, même si tu soumets `pubs=[pub1, pub2]`, rien ne garantit que les circuits de `pub1` s'exécuteront avant ceux de `pub2`. Il n'y a pas non plus de garantie que les circuits du même job s'exécuteront en un seul batch sur la QPU.

## Spécifier rep_delay pour un job primitif

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()

# Make sure your backend supports it
backend = service.least_busy(
    operational=True, min_num_qubits=100, dynamic_reprate_enabled=True
)

# Determine the allowable range
backend.rep_delay_range
sampler = Sampler(mode=backend)

# Specify a value in the supported range
sampler.options.execution.rep_delay = 0.0005

## Étapes suivantes
> **Tip:** - Essaie un exemple dans le tutoriel [Algorithme d'optimisation approximative quantique (QAOA)](/tutorials/quantum-approximate-optimization-algorithm).
>   - Consulte comment [démarrer avec les primitives.](./get-started-with-primitives)